# Video Dataset Exploration

Initial dataset survey for the video pipeline: build a per-class subdirectory tree under `DATASET_COLOR_BODY_WITHOUT/`, count clips per sign, plot duration / FPS histograms, and prototype the ViViT preprocessing pipeline (frame extraction, image-processor batching). Most of the production logic later moved into `videos/scripts/build_dataset.py` and `videos/src/dataset.py`; this notebook is preserved for the EDA plots and the reorganized-by-class directory copy.

## Configuration

In [ ]:
import os

BASE_DIR = os.path.split(os.getcwd())[0]
VIDEOS_DIR = os.path.join(BASE_DIR, "COLOR_BODY")
NEW_DATASET_DIR = os.path.join(BASE_DIR, "DATASET_COLOR_BODY_WITHOUT")
VIVIT_CHECKPOINT = "google/vivit-b-16x2-kinetics400"
MAX_FRAMES = 16
FRAME_SIZE = (224, 224)


## Setup

In [ ]:
from transformers import VivitForVideoClassification, VivitImageProcessor, Trainer, TrainingArguments
import torch
import os, glob, shutil
import matplotlib.pyplot as plt
from datasets import DatasetDict, Dataset
import cv2, tqdm
import numpy as np

## Reorganize clips into per-class subfolders

In [ ]:
base_dir = BASE_DIR

videos_dir = VIDEOS_DIR

new_dataset_dir = NEW_DATASET_DIR

"""if not os.path.exists(new_dataset_dir):
    os.mkdir(new_dataset_dir)

for file_route in glob.glob(os.path.join(videos_dir,"*.avi")):
    video_name = os.path.split(file_route)[1]
    sign, volunteer, rep = video_name[:len(video_name) - 4].split("_")
    if not os.path.exists(os.path.join(new_dataset_dir,str(sign))):
        os.mkdir(os.path.join(new_dataset_dir,str(sign)))
    
    shutil.copyfile(file_route,os.path.join(new_dataset_dir,str(sign),f"{volunteer}_{rep}.avi"))

"""

## Clip count per sign

In [ ]:
info = {}
duration = []
fps = []
for folder in glob.glob(os.path.join(new_dataset_dir,"*")):
    direct, sign = os.path.split(folder)
    info[sign] = len(glob.glob(os.path.join(folder,"*.avi")))
    for video in glob.glob(os.path.join(folder,"*.avi")):
        cap = cv2.VideoCapture(video)
        duration.append(cap.get(cv2.CAP_PROP_FRAME_COUNT) / cap.get(cv2.CAP_PROP_FPS))
        fps.append(cap.get(cv2.CAP_PROP_FPS))
        cap.release()


plt.figure(figsize=(10,10))
plt.bar(list(info.keys()),list(info.values()))
plt.title("Videos per sign")
plt.xlabel("Sign")
plt.ylabel("# of videos")
plt.xticks(rotation=90)
plt.show()


## Duration distribution

In [ ]:
plt.figure(figsize=(10,10))
plt.hist(duration, bins=100)
plt.vlines(np.mean(duration), 0, 50, colors="red", label="Mean")
plt.text(np.mean(duration), 50, f"Mean: {round(np.mean(duration),2)}")
plt.vlines(np.min(duration), 0, 50, colors="green", label="Min")
plt.text(np.min(duration), 50, f"Min: {round(np.min(duration),2)}")
plt.vlines(np.max(duration), 0, 50, colors="blue", label="Max")
plt.text(np.max(duration), 50, f"Max: {round(np.max(duration),2)}")
plt.vlines(np.mean(duration) + np.std(duration), 0, 50, colors="orange", label="Mean + Std")
plt.text(np.mean(duration) + np.std(duration), 50, f"Std: {round(np.std(duration),2)}")
plt.vlines(np.mean(duration) - np.std(duration), 0, 50, colors="orange", linestyles="dashed", label="Mean - Std")
plt.legend()
plt.title("Duration of videos")
plt.xlabel("Duration (seconds)")
plt.ylabel("Count")
plt.show()

plt.figure(figsize=(10,10))
plt.hist(fps,bins=100)
plt.vlines(np.mean(fps),0,50,colors="red",label="Mean")
plt.text(np.mean(fps),50,f"Mean: {round(np.mean(fps),2)}")
plt.title("Duration of videos")
plt.xlabel("Duration (seconds)")
plt.ylabel("Count")
plt.show()

## Frame extraction prototype

In [ ]:

def extract_frames(video_path, max_frames=16, size=(224, 224)):
    cap = cv2.VideoCapture(video_path)
    frames = []
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    frame_indices = torch.linspace(0, total_frames - 1, steps=max_frames).long()
    for i in range(total_frames):
        ret, frame = cap.read()
        if i in frame_indices and ret:
            frame = cv2.resize(frame, size)
            rgb_frame = cv2.cvtColor(frame,cv2.COLOR_BGR2RGB)
            f_frame = np.transpose(rgb_frame, (2, 0, 1))
            frames.append(f_frame.astype("float32")/255.0)
        if len(frames) == max_frames:
            break
    cap.release()
    return frames

def extract_frames_without_sampling(video_path):
    cap = cv2.VideoCapture(video_path)
    frames = []
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    #frame_indices = torch.linspace(0, total_frames - 1, steps=max_frames).long()
    for i in range(total_frames):
        ret, frame = cap.read()
       
        #frame = cv2.resize(frame, size)
        rgb_frame = cv2.cvtColor(frame,cv2.COLOR_BGR2RGB)
        f_frame = np.transpose(rgb_frame, (2, 0, 1))
        frames.append(f_frame.astype("float32")/255.0)

    cap.release()
    return frames

def prepare_data(folder, max_frames=32):
    data = {"video": [], "label": []}
    label_to_id = {label: i for i, label in enumerate(sorted(os.listdir(folder)))}
    labels = os.listdir(folder)
    for label in tqdm.tqdm(range(len(labels))):
        class_dir = os.path.join(folder, labels[label])
        if os.path.isdir(class_dir):
            for video_file in os.listdir(class_dir):
                video_path = os.path.join(class_dir, video_file)
                frames = extract_frames(video_path, max_frames)
                if frames: 
                    data["video"].append(frames)
                    data["label"].append(label_to_id[labels[label]])
    return data

def prepare_data_without_sampling(folder):
    data = {"video": [], "label": []}
    label_to_id = {label: i for i, label in enumerate(sorted(os.listdir(folder)))}
    labels = os.listdir(folder)
    for label in tqdm.tqdm(range(len(labels))):
        class_dir = os.path.join(folder, labels[label])
        if os.path.isdir(class_dir):
            for video_file in os.listdir(class_dir):
                video_path = os.path.join(class_dir, video_file)
                frames = extract_frames_without_sampling(video_path)
                if frames: 
                    data["video"].append(frames)
                    data["label"].append(label_to_id[labels[label]])
    return data

def custom_train_test_split(dataset, test_split=0.2):

    train_ds = {"video":[],"label":[]}
    test_ds = {"video":[],"label":[]}

    imgs_per_class = len(dataset["label"])//len(np.unique(dataset["label"]))
    test_imgs_per_class = int(imgs_per_class*test_split)
    labels = np.array(dataset["label"])
    for class_indx in tqdm.tqdm(np.unique(dataset["label"])):
        img_indx = np.random.permutation(np.where(labels == class_indx)[0])

        test_indx = img_indx[:test_imgs_per_class]
        train_indx = img_indx[test_imgs_per_class:]

        for i in train_indx:
            train_ds["video"].append(dataset["video"][i])
            train_ds["label"].append(labels[i])
        
        for j in test_indx:
            test_ds["video"].append(dataset["video"][j])
            test_ds["label"].append(labels[j])

    return train_ds, test_ds
    
    


## Build train/test splits

In [ ]:
dataset_dict = prepare_data(new_dataset_dir)

In [ ]:
train_ds, test_ds = custom_train_test_split(dataset_dict)

## Image-processor wiring

In [ ]:
# Convert to Hugging Face Dataset
dataset = DatasetDict({
    "train": Dataset.from_dict(train_ds),
    "test": Dataset.from_dict(test_ds),
})


## Preprocess function prototype

In [ ]:
processor = VivitImageProcessor.from_pretrained(VIVIT_CHECKPOINT,
                                                data_format="ChannelDimension.FIRST",
                                                )



In [ ]:
def preprocess_function(examples):
    processed_ds = {"pixel_values":[],"labels":[]}
    video = torch.tensor(examples["video"], dtype=torch.float32)  # Convert to float32 tensor
    for batch in range(video.shape[0]):

        vid = []
        video_batch = video[batch]
        for frame in range(video_batch.shape[0]):

            processed = processor(video_batch[frame], return_tensors="pt")

            vid.append(processed["pixel_values"][0,0])

        processed_ds["pixel_values"].append(torch.stack(vid))
    
    processed_ds["pixel_values"] = torch.stack(processed_ds["pixel_values"])
    processed_ds["labels"] = examples["label"]
    return processed_ds

processed_dataset = dataset.map(preprocess_function, batched=True, batch_size=16)
processed_dataset.set_format(type="torch", columns=["pixel_values", "labels"])
processed_dataset.save_to_disk("dataset")